<a href="https://colab.research.google.com/github/wadaka0821/nlp-tutorial/blob/main/questions/3_5_word2vec_implementation_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install datasets
!pip install gensim
!pip uninstall -y numpy
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [1]:
import numpy as np

print(np.__version__)

1.26.4


In [3]:
from datasets import load_dataset
import nltk
import gensim
nltk.download('punkt_tab')

seed = 42

dataset = load_dataset("wikipedia", "20220301.simple")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
# テキストを取り出す
texts = [dataset['train']['text'][i] for i in range(500)]
# 単語分割
texts_tokenized = [nltk.word_tokenize(text.lower()) for text in texts if text]

In [5]:
import torch

In [6]:
# Pytorchのシード値を固定
# 必要に応じてpythonやnumpyなどのシード値も固定する必要があります（再現性を持たせたい場合）
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
# 使用可能なデバイスの確認
f'{device=}'

"device='cuda'"

In [8]:
class Word2Vec(torch.nn.Module):
  def __init__(self, corpus, dim=100):
    super(Word2Vec, self).__init__()

    self.dim = dim
    self.vocab = dict()
    self.vocab_size = 0
    print('start building vocabulary')
    self.build_vocab(corpus)
    print(f'finish building vocabulary\nvocabulary size is {self.vocab_size}')
    self.embedding = torch.nn.Linear(self.vocab_size, dim, bias=False)
    self.output_layer = torch.nn.Linear(dim, self.vocab_size, bias=False)

  def build_vocab(self, corpus):
    for sent in corpus:
      for word in sent:
        if word not in self.vocab:
          self.vocab[word] = len(self.vocab)
    self.vocab_size = len(self.vocab)

  def word2id(self, x):
    ids = [self.vocab[i] for i in x]
    return ids

  def forward(self, x):
    x = torch.nn.functional.one_hot(x, num_classes=self.vocab_size)
    h = self.embedding(x.float())
    logits = self.output_layer(h)

    return logits

In [9]:
N = 10000
model = Word2Vec(texts_tokenized[:N])

start building vocabulary
finish building vocabulary
vocabulary size is 23850


In [10]:
window = 2
x, y = list(), list()
for sent in texts_tokenized[:N]:
  for i in range(len(sent)):
    tmp_y = sent[max(0, i-window):i]
    if i < len(sent)-1:
      tmp_y += sent[min(i+1, len(sent)):min(i+window, len(sent))]
    y += tmp_y
    x += [sent[i] for _ in range(len(tmp_y))]

In [11]:
len(x)

1149082

In [12]:
x_ids = model.word2id(x)
y_ids = model.word2id(y)

In [13]:
batch_size = 512

datasets = torch.utils.data.TensorDataset(torch.tensor(x_ids), torch.tensor(y_ids))
dataloader = torch.utils.data.DataLoader(datasets, batch_size=batch_size)

In [14]:
from tqdm import tqdm

In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
MAX_EPOCH = 5
criterion = torch.nn.CrossEntropyLoss()

model.to(device)

for epoch in range(1, MAX_EPOCH+1):
  for it, batch in tqdm(enumerate(dataloader, 1), total=len(dataloader)):
    optimizer.zero_grad()

    pred = model(batch[0].to(device))
    loss = criterion(pred, batch[1].to(device))
    loss.backward()

    optimizer.step()

    # print(f'{epoch=}, {it=}, loss={loss.item():.5f}', end='\r')

100%|██████████| 2245/2245 [00:18<00:00, 118.88it/s]


In [16]:
def most_similar(model, word):
  model.to('cpu')
  sims = list()
  with torch.no_grad():
    x = torch.nn.functional.one_hot(torch.tensor(model.vocab[word]), num_classes=model.vocab_size)
    word_vec = model.embedding(x.float())
    sims = torch.matmul(model.embedding.weight.T, word_vec) / torch.linalg.norm(word_vec) / torch.linalg.norm(model.embedding.weight.T, dim=1)
    # for i in model.vocab.keys():
    #   if i == word:
    #     continue
    #   vec = torch.nn.functional.one_hot(torch.tensor(model.vocab[i]), num_classes=model.vocab_size)
    #   vec = model.embedding(vec.float())
    #   sims.append([i, torch.nn.CosineSimilarity()(word_vec.view(1, -1), vec.view(1, -1))])
  return sims

In [17]:
torch.topk(most_similar(model, 'translation'), k=6)

torch.return_types.topk(
values=tensor([1.0000, 0.4968, 0.4966, 0.4802, 0.4585, 0.4541]),
indices=tensor([12089,   832, 17669, 16330, 22121, 12144]))

In [18]:
for i, j in model.vocab.items():
  # 1個下の行のリストには、上の出力で得たindicsのリストに書き換えてください。
  if j in [12091, 22124, 16470, 12146, 23757,  3587]:
    print(i)

assigned
licensing
deductive
contested
xml
ipv6


In [19]:
torch.topk(most_similar(model, 'language'), k=6)

torch.return_types.topk(
values=tensor([1.0000, 0.5362, 0.4858, 0.4193, 0.4178, 0.4149]),
indices=tensor([ 224, 1693,   97, 1613, 1094,   63]))

In [20]:
for i, j in model.vocab.items():
  if j in [ 224, 1693,   97, 8186, 2581, 2715]:
    print(i)

word
language
languages
continent
point
context


In [21]:
torch.topk(most_similar(model, 'sentiment'), k=6)

torch.return_types.topk(
values=tensor([1.0000, 0.5190, 0.5144, 0.5077, 0.5036, 0.4927]),
indices=tensor([18439, 20498,  8873, 17920, 19662, 18017]))

In [22]:
for i, j in model.vocab.items():
  if j in [18442, 17810, 17923, 16805, 17963, 12371]:
    print(i)

globe
reptile
button
korrect
multi-tasking
1919.


## 問題1
---
今回実装してあるプログラムでは類似度の高い単語を得る most_similar 関数のみです．実際の Word2Vec のモデルは足し引きも可能です．これを実装してください．

## 問題2
---
今回は word2vec の学習に skip-gram法を使用しました．別の方法として，CBOWという方法もあります．この CBOW による実装をしてみてください．